# ⚽ TacticsAI — YOLOv8 Fine-tuning (Google Colab)

Fine-tunes YOLOv8x on a football-specific dataset from Roboflow.
At the end you'll download `best.pt` — the fine-tuned model for TacticsAI.

**Runtime:** GPU → Runtime → Change runtime type → T4 GPU (free)

**Time:** ~2 hours for 50 epochs on T4 | ~45 min on A100

---
| Step | Description |
|---|---|
| 1 | Install dependencies |
| 2 | Mount Google Drive (for saving checkpoints) |
| 3 | Download football dataset from Roboflow |
| 4 | Fine-tune YOLOv8x |
| 5 | Evaluate results |
| 6 | Download best.pt |

## Step 1 — Install dependencies

In [ ]:
# Verify GPU
!nvidia-smi
print()

# Install Ultralytics (YOLOv8) and Roboflow SDK
!pip install ultralytics roboflow supervision --quiet

import ultralytics
ultralytics.checks()   # prints CUDA info + version

## Step 2 — Mount Google Drive
Optional but recommended — saves training checkpoints to Drive so they survive session restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/TacticsAI'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {SAVE_DIR}')

## Step 3 — Download football dataset from Roboflow

Dataset: **Football Player Detection** (~5,000 annotated broadcast images)
- Source: [roboflow.com/football-player-detection](https://roboflow.com/)
- Classes: `player`, `goalkeeper`, `referee`, `ball`
- Free to use (Roboflow Universe)

**Get your free API key:** [app.roboflow.com](https://app.roboflow.com) → Settings → API Keys

In [ ]:
from roboflow import Roboflow
import os

# ────────────────────────────────────────────────
# Replace with your Roboflow API key
# Get it free at: https://app.roboflow.com
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'
# ────────────────────────────────────────────────

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Football Player Detection dataset (update version if needed)
project = rf.workspace('roboflow-jvuqo').project('football-players-detection-3zvbc')
dataset  = project.version(1).download('yolov8')

print(f'\nDataset downloaded to: {dataset.location}')
print(f'YAML config: {dataset.location}/data.yaml')

### (Alternative) Use the DFL dataset — no API key needed
If the Roboflow download fails, you can use this pre-exported version:

In [ ]:
# Alternative: Download pre-exported dataset directly
# Uncomment and run this cell if the Roboflow API step fails

# !wget -q https://storage.googleapis.com/tacticsai-public/football-dataset.zip
# !unzip -q football-dataset.zip -d football-dataset
# dataset_yaml = 'football-dataset/data.yaml'
# print('Alternative dataset ready.')

## Step 4 — Fine-tune YOLOv8x

**Key hyperparameters:**
- `imgsz=1280` — high resolution captures small distant players
- `epochs=50` — sufficient for fine-tuning (base model already knows 'person')
- `batch=8` — fits in T4 16GB VRAM at 1280px
- `freeze=10` — freeze first 10 backbone layers (faster, avoids overfitting)

In [ ]:
from ultralytics import YOLO
import os

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_YAML  = f'{dataset.location}/data.yaml'   # update if using alternative
EPOCHS        = 50
IMAGE_SIZE    = 1280
BATCH_SIZE    = 8
FREEZE_LAYERS = 10    # freeze backbone to speed up training
PROJECT_NAME  = 'TacticsAI'
RUN_NAME      = 'yolov8x-football-v1'
# ─────────────────────────────────────────────────────────────────────────────

# Load base model (auto-downloads ~130MB if not cached)
model = YOLO('yolov8x.pt')

print(f'Starting fine-tuning: {EPOCHS} epochs @ {IMAGE_SIZE}px')
print(f'Dataset: {DATASET_YAML}\n')

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    freeze=FREEZE_LAYERS,
    device='cuda',
    project=PROJECT_NAME,
    name=RUN_NAME,
    patience=15,          # early stopping
    save=True,
    save_period=10,       # checkpoint every 10 epochs
    plots=True,           # training curves
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,          # cosine LR schedule
    augment=True,
    mosaic=1.0,
    mixup=0.1,
)

## Step 5 — Evaluate results

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

# Path to best weights
best_model_path = f'{PROJECT_NAME}/{RUN_NAME}/weights/best.pt'
print(f'Best model: {best_model_path}')

# Run validation
best_model = YOLO(best_model_path)
val_results = best_model.val(data=DATASET_YAML, imgsz=IMAGE_SIZE)

print(f'\n── Validation Results ──────────────────')
print(f'mAP50     : {val_results.box.map50:.4f}')
print(f'mAP50-95  : {val_results.box.map:.4f}')
print(f'Precision : {val_results.box.mp:.4f}')
print(f'Recall    : {val_results.box.mr:.4f}')
print(f'────────────────────────────────────────')

# Show training curves
results_dir = Path(f'{PROJECT_NAME}/{RUN_NAME}')
curve_path = results_dir / 'results.png'
if curve_path.exists():
    display(Image(str(curve_path)))

# Show confusion matrix
cm_path = results_dir / 'confusion_matrix.png'
if cm_path.exists():
    display(Image(str(cm_path)))

In [ ]:
# Run inference on a sample validation image
from pathlib import Path
import glob

val_images = glob.glob(f'{dataset.location}/valid/images/*.jpg')[:3]

if val_images:
    pred = best_model.predict(
        val_images,
        conf=0.5,
        imgsz=IMAGE_SIZE,
        save=True,
        project='predictions',
        name='val_samples',
    )
    # Display result images
    for img_path in glob.glob('predictions/val_samples/*.jpg'):
        display(Image(img_path, width=800))
else:
    print('No validation images found — check dataset path.')

## Step 6 — Download best.pt

After downloading, place `best.pt` in the root of your TacticsAI project directory.
Then in `app.py` sidebar, select **best.pt (fine-tuned on football)**.

In [ ]:
import shutil

# Copy best.pt to Google Drive
best_src = f'{PROJECT_NAME}/{RUN_NAME}/weights/best.pt'
best_dst = f'{SAVE_DIR}/best.pt'
shutil.copy(best_src, best_dst)
print(f'Saved to Drive: {best_dst}')

In [ ]:
# Download best.pt directly to your computer
from google.colab import files

files.download(f'{PROJECT_NAME}/{RUN_NAME}/weights/best.pt')
print('Download started — check your browser Downloads folder.')
print()
print('Next step: place best.pt in c:\\TacticsAI\\ and run the dashboard!')

---

## After downloading best.pt

1. Place `best.pt` in `c:\TacticsAI\`
2. Activate your venv: `.\venv\Scripts\Activate.ps1`
3. Run dashboard: `streamlit run app.py`
4. In the sidebar, select **best.pt (fine-tuned on football)**

---
**Expected mAP50** after 50 epochs on T4: **~0.85–0.92**

Better than base YOLOv8 for football because:
- Trained on broadcast camera angles specifically
- Separate classes for player / goalkeeper / referee / ball
- Filters out photographers and crowd